# Reporting

In [61]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import sys 
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Read Counterfactuals Config File

In [62]:
config_path = Path(r'experiments')
config_filename =  "bin_cf_final.yml"
config_dict = ymlconfig.load_config(config_path / config_filename)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict

{'experiment': {'summary': 'binary classification - Counterfactual Analysis (Final- No NCS, Sudo)',
  'classification_type': 'binary',
  'stage': 'counterfactuals',
  'tag': 'final_no_unactionable',
  'verbosity': 0,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'model': {'code': 'catboost', 'name': 'CatBoost'},
 'explainability': {'ksplit_trained_model_results_file': 'binary\\explainability\\catboost\\final\\catboost_ksplit_trained_models.joblib',
  'rundate': '2026-03-18',
  'tag': 'final'},
 'dice': {'method': 'genetic',
  'cf_features': {'actionable': 'INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI',
   'unactionable': 'SEX,AGE,SUBJ,DM_DUR,GBS',
   'progressive': 'none'},
  'global_cf': {'total_CFs': 20, 'posthoc_sparsity_algorithm': 'binary'},
  'local_cf': {'nrepeats': 3,
   'total_CFs': 20,
   'posthoc_sparsity_algorithm': 'binary',
   'posthoc_sparsity_param': 0.05,
   'proximity_weight': 0.5,
   'd

## Set Directories

In [63]:
nsplits = 3

outputdir = config_path /  config.experiment.classification_type /  config.experiment.stage / config.model.code / config.experiment.tag 
assert outputdir.is_dir()

split_output_dirs = []
for midx in range(nsplits):
    s = outputdir / f'split{midx}'
    assert s.is_dir()
    split_output_dirs.append(s)

## Instances of Interest

In [64]:
outputdir, split_output_dirs

(WindowsPath('experiments/binary/counterfactuals/catboost/final_no_unactionable'),
 [WindowsPath('experiments/binary/counterfactuals/catboost/final_no_unactionable/split0'),
  WindowsPath('experiments/binary/counterfactuals/catboost/final_no_unactionable/split1'),
  WindowsPath('experiments/binary/counterfactuals/catboost/final_no_unactionable/split2')])

In [65]:
ioi = {}
for midx in range(nsplits):
    filename = f'{config.model.code}_split{midx}_instances_of_interest.csv'
    df = pd.read_csv(split_output_dirs[midx] / filename)
    ioi[midx] = {
        'Model': f'Split {midx}', 
        'Instances': df.shape[0], 
        'Misclassified': df.misclassified.sum()
        }
ioi

{0: {'Model': 'Split 0', 'Instances': 6, 'Misclassified': 4},
 1: {'Model': 'Split 1', 'Instances': 7, 'Misclassified': 4},
 2: {'Model': 'Split 2', 'Instances': 9, 'Misclassified': 6}}

In [66]:
ioi_df = pd.DataFrame(ioi).T
ioi_df

,Model,Instances,Misclassified
0,Split 0,6,4
1,Split 1,7,4
2,Split 2,9,6


## Model 0 Summary

In [67]:
midx = 0
ioi_filename = f'{config.model.code}_split{midx}_instances_of_interest.csv'
ioi_df = pd.read_csv(split_output_dirs[midx] / ioi_filename)
ioi_df = ioi_df.rename(columns={'Unnamed: 0':'ID'})
ioi_df

,ID,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,pred_proba,margin,pred,actual,misclassified
0,4,1,57.0,0,5.0,1.0,14.4,0,0,0,0,0,1,0,1,1,1.0,4.19,41.90,3.70,38.20,38.5,4.50,8.89,6.88,48.3,5.36,45.5,4.42,39.50,38.3,4.00,10.29,8.82,49.9,54.0,3.0,63.0,0.0,54.0,36.0,0.719,0.083,1,1,False
1,55,0,67.0,1,11.0,0.0,11.1,1,0,1,0,0,1,1,1,1,4.0,8.74,47.50,4.03,44.50,45.2,4.85,14.13,9.15,49.2,7.24,46.5,7.04,53.10,42.5,4.00,8.42,6.23,50.0,67.0,2.0,49.0,9.0,47.0,32.0,0.767,0.131,1,1,False
2,67,1,56.0,1,0.0,0.0,5.1,0,0,0,0,0,0,0,0,0,3.0,16.27,52.00,12.16,50.70,46.6,3.75,9.46,6.93,47.0,13.80,50.0,13.48,53.00,43.8,3.20,7.41,5.61,47.7,71.0,8.0,63.0,6.0,60.0,30.0,0.187,0.449,0,1,True
3,106,1,33.0,1,3.0,1.0,9.0,0,0,0,0,0,0,1,0,1,3.0,4.88,49.20,0.00,0.00,43.2,3.50,13.14,10.75,43.3,2.68,45.3,0.00,0.00,44.0,3.45,9.98,7.18,42.4,76.0,0.0,76.0,12.0,90.0,4.0,0.524,0.111,0,1,True
4,128,0,74.0,1,30.0,0.0,4.9,1,0,1,0,0,0,1,1,1,4.0,14.40,47.60,11.03,45.50,45.1,3.95,10.29,7.78,46.4,7.45,46.4,11.65,48.50,42.3,3.45,7.71,5.12,46.6,46.0,0.0,60.0,0.0,34.0,30.0,0.275,0.360,0,1,True
5,158,0,43.0,1,2.0,0.0,5.2,0,0,1,0,0,0,0,1,0,9.0,12.19,1.96,9.78,2.38,47.2,4.15,14.75,8.91,43.3,34.31,57.7,19.70,2.58,49.3,3.45,15.36,9.43,42.9,57.0,6.0,50.0,7.0,70.0,29.0,0.037,0.598,0,1,True


In [68]:
ioi_df[ioi_df.ID==4].loc[0].margin

0.0833982898440675

In [69]:
cf = {}
for qidx in ioi_df.ID.values:
    instance = ioi_df[ioi_df.ID==qidx].iloc[0]
    qidx_filename = f'{config.model.code}_split{midx}_local_cf_distances.csv'
    qidx_df = pd.read_csv(split_output_dirs[midx] / 'nofiltering' / str(qidx).zfill(3) / qidx_filename)
    cf_count = qidx_df.shape[0]    
    cf[qidx] = {
        'ID': qidx, 
        'probability': instance.pred_proba,
        'margin': instance.margin,
        'prediction': 'Confirmed' if instance.pred else 'Unconfirmed',
        'actual': 'Confirmed' if instance.actual else 'Unconfirmed',
        'misclassified': 'Yes' if instance.misclassified else 'No',
        'CF count': cf_count, 
        'sparsity': np.nan if not cf_count else qidx_df['sparsity'].mean(),
        'L1 mean': np.nan if not cf_count else qidx_df['L1_dist'].mean(),
        'L2 mean': np.nan if not cf_count else qidx_df['L2_dist'].mean(),
        }
cf_df = pd.DataFrame(cf).T.reset_index(drop=True)
cf_df['ID'] = cf_df['ID'].astype(int)
cf_df['CF count'] = cf_df['CF count'].astype(int)
cf_df

,ID,probability,margin,prediction,actual,misclassified,CF count,sparsity,L1 mean,L2 mean
0,4,0.719,0.083,Confirmed,Confirmed,No,4,28.25,116.995,30.669
1,55,0.767,0.131,Confirmed,Confirmed,No,22,27.545,107.345,29.713
2,67,0.187,0.449,Unconfirmed,Confirmed,Yes,20,24.0,92.632,28.135
3,106,0.524,0.111,Unconfirmed,Confirmed,Yes,4,25.25,114.363,34.028
4,128,0.275,0.36,Unconfirmed,Confirmed,Yes,35,22.829,93.903,28.71
5,158,0.037,0.598,Unconfirmed,Confirmed,Yes,35,26.143,112.511,31.527


## Full Script

In [102]:
ioi_summary_dfs = []
gi_dfs = []
cf_dfs = {}
for midx in range(nsplits):
    ioi = {}

    # Read Instance of Interest
    ioi_filename = f'{config.model.code}_split{midx}_instances_of_interest.csv'
    ioi_df = pd.read_csv(split_output_dirs[midx] / ioi_filename)
    ioi_df = ioi_df.rename(columns={'Unnamed: 0':'ID'})
    
    # Read Global Importance
    gi_filename = f'{config.model.code}_split{midx}_global_importance.csv'
    gi_df = pd.read_csv(split_output_dirs[midx] / gi_filename)
    gi_df = gi_df.rename(columns={'Unnamed: 0':'Feature'})
    gi_df = gi_df.rename(columns={'0':f'Split {midx}'})
    gi_dfs.append(gi_df)

    # Process Instance Folders
    cf = {}
    for qidx in ioi_df.ID.values:
        instance = ioi_df[ioi_df.ID==qidx].iloc[0]        
        qidx_filename = f'{config.model.code}_split{midx}_local_cf_distances.csv'        
        qidx_path = split_output_dirs[midx] / 'nofiltering' / str(qidx).zfill(3) / qidx_filename
        if not qidx_path.is_file():
            continue
        qidx_df = pd.read_csv(qidx_path)
        cf_count = qidx_df.shape[0]    
        cf[qidx] = {
            'model': midx, 
            'ID': qidx, 
            'prediction': 'Confirmed' if instance.pred else 'Unconfirmed',
            'actual': 'Confirmed' if instance.actual else 'Unconfirmed',
            'misclassified': 'Yes' if instance.misclassified else 'No',            
            'probability': instance.pred_proba,
            'margin': instance.margin,
            'CF Count': cf_count, 
            'Sparsity': np.nan if not cf_count else qidx_df['sparsity'].mean(),
            'L1 Mean': np.nan if not cf_count else qidx_df['L1_dist'].mean(),
            'L2 Mean': np.nan if not cf_count else qidx_df['L2_dist'].mean(),
            }
        
    # Counterfactual Stats for this Instance
    if not cf:
        print(f'No counterfactuals found for {midx}')
        continue
    cf_df = pd.DataFrame(cf).T.reset_index(drop=True)
    cf_df['ID'] = cf_df['ID'].astype(int)
    cf_df['CF Count'] = cf_df['CF Count'].astype(int)
    cf_dfs[midx] = cf_df

    # Instance of Interest for this Model
    ioi[midx] = {
        'Model': f'Split {midx}', 
        'Instances': ioi_df.shape[0], 
        'Misclassified': ioi_df.misclassified.sum(),
        'Instances with CF': (cf_df['CF Count'] > 0).sum(),
        'Sparsity Mean': cf_df['Sparsity'].mean(),
        'L1 Mean': cf_df['L1 Mean'].mean(),
        'L2 Mean': cf_df['L2 Mean'].mean(),
        }    
    ioi_summary_dfs.append(pd.DataFrame(ioi).T)

## Concatenate results for models

# Global Importance
gi_summary_df = gi_dfs[0].merge(gi_dfs[1], on='Feature').merge(gi_dfs[2], on='Feature')
gi_summary_df['Mean'] = gi_summary_df[['Split 0', 'Split 1', 'Split 2']].mean(axis=1)
# gi_summary_df['Std'] = gi_summary_df[['Split 0', 'Split 1', 'Split 2']].std(axis=1)
gi_summary_df = gi_summary_df.sort_values(by='Mean', ascending=False)

# Instance of Interest Summary
ioi_summary_df = pd.concat(ioi_summary_dfs, axis=0)
ioi_summary_df

,Model,Instances,Misclassified,Instances with CF,Sparsity Mean,L1 Mean,L2 Mean
0,Split 0,6,4,6,25.669,106.291,30.464
1,Split 1,7,4,6,25.97,104.365,29.863
2,Split 2,9,6,8,25.543,101.207,29.327


### Global Importance Summary

In [103]:
gi_summary_df

,Feature,Split 0,Split 1,Split 2,Mean
3,SSA_L,9.990e-01,1.000,1.000,9.997e-01
0,CMAPKNE_L,1.000e+00,0.999,1.000,9.996e-01
2,CMAPANK_L,9.990e-01,1.000,0.999,9.994e-01
1,CMAPKNE_R,1.000e+00,1.000,0.996,9.988e-01
4,CMAPANK_R,9.971e-01,0.999,0.998,9.980e-01
6,FWAVE_L,9.961e-01,0.996,0.993,9.951e-01
8,MCV_L,9.942e-01,0.991,0.996,9.939e-01
7,FWAVE_R,9.961e-01,0.993,0.992,9.937e-01
13,SSA_R,9.835e-01,0.998,0.996,9.925e-01
5,SPSA_L,9.971e-01,0.985,0.994,9.920e-01


### IOI Summary

In [104]:
ioi_summary_dfs[0]

,Model,Instances,Misclassified,Instances with CF,Sparsity Mean,L1 Mean,L2 Mean
0,Split 0,6,4,6,25.669,106.291,30.464


### Model 0 Summary

In [105]:
cf_dfs[0] 

,model,ID,prediction,actual,misclassified,probability,margin,CF Count,Sparsity,L1 Mean,L2 Mean
0,0,4,Confirmed,Confirmed,No,0.719,0.083,4,28.25,116.995,30.669
1,0,55,Confirmed,Confirmed,No,0.767,0.131,22,27.545,107.345,29.713
2,0,67,Unconfirmed,Confirmed,Yes,0.187,0.449,20,24.0,92.632,28.135
3,0,106,Unconfirmed,Confirmed,Yes,0.524,0.111,4,25.25,114.363,34.028
4,0,128,Unconfirmed,Confirmed,Yes,0.275,0.36,35,22.829,93.903,28.71
5,0,158,Unconfirmed,Confirmed,Yes,0.037,0.598,35,26.143,112.511,31.527


### Model 1 Summary

In [106]:
cf_dfs[1] 

,model,ID,prediction,actual,misclassified,probability,margin,CF Count,Sparsity,L1 Mean,L2 Mean
0,1,12,Confirmed,Confirmed,No,0.666,0.031,38,27.605,111.169,30.8
1,1,54,Confirmed,Confirmed,No,0.751,0.116,45,28.267,121.284,32.891
2,1,73,Confirmed,Unconfirmed,Yes,0.934,0.298,39,28.308,122.743,32.733
3,1,94,Unconfirmed,Unconfirmed,No,0.481,0.154,11,18.364,56.315,20.65
4,1,119,Confirmed,Unconfirmed,Yes,0.725,0.09,25,26.36,110.292,31.537
5,1,178,Confirmed,Unconfirmed,Yes,0.701,0.066,35,26.914,104.387,30.569


### Model 2 Summary

In [107]:
cf_dfs[2] 

,model,ID,prediction,actual,misclassified,probability,margin,CF Count,Sparsity,L1 Mean,L2 Mean
0,2,22,Unconfirmed,Confirmed,Yes,0.171,0.464,31,25.161,94.326,28.951
1,2,45,Confirmed,Unconfirmed,Yes,0.9,0.265,15,25.933,107.987,30.173
2,2,68,Unconfirmed,Confirmed,Yes,0.007,0.629,25,24.16,114.459,34.346
3,2,88,Confirmed,Confirmed,No,0.826,0.191,19,25.684,92.409,26.595
4,2,96,Unconfirmed,Confirmed,Yes,0.512,0.123,23,25.478,94.64,27.299
5,2,136,Confirmed,Confirmed,No,0.648,0.013,12,24.25,91.867,28.033
6,2,155,Confirmed,Confirmed,No,0.745,0.11,8,30.875,136.09,35.099
7,2,167,Unconfirmed,Confirmed,Yes,0.127,0.508,25,22.8,77.874,24.117


In [108]:
cf_dfs_stacked = pd.concat([cf_dfs[i] for i in range(3)], axis=0)
cf_dfs_stacked

,model,ID,prediction,actual,misclassified,probability,margin,CF Count,Sparsity,L1 Mean,L2 Mean
0,0,4,Confirmed,Confirmed,No,0.719,0.083,4,28.25,116.995,30.669
1,0,55,Confirmed,Confirmed,No,0.767,0.131,22,27.545,107.345,29.713
2,0,67,Unconfirmed,Confirmed,Yes,0.187,0.449,20,24.0,92.632,28.135
3,0,106,Unconfirmed,Confirmed,Yes,0.524,0.111,4,25.25,114.363,34.028
4,0,128,Unconfirmed,Confirmed,Yes,0.275,0.36,35,22.829,93.903,28.71
5,0,158,Unconfirmed,Confirmed,Yes,0.037,0.598,35,26.143,112.511,31.527
0,1,12,Confirmed,Confirmed,No,0.666,0.031,38,27.605,111.169,30.8
1,1,54,Confirmed,Confirmed,No,0.751,0.116,45,28.267,121.284,32.891
2,1,73,Confirmed,Unconfirmed,Yes,0.934,0.298,39,28.308,122.743,32.733
3,1,94,Unconfirmed,Unconfirmed,No,0.481,0.154,11,18.364,56.315,20.65


### Produce LaTeX Tables

#### Global Importances

In [97]:
print(gi_summary_df.to_latex(index=False, float_format="{:.3f}".format))

\begin{tabular}{lrrrr}
\toprule
Feature & Split 0 & Split 1 & Split 2 & Mean \\
\midrule
SSA_L & 0.999 & 1.000 & 1.000 & 1.000 \\
CMAPKNE_L & 1.000 & 0.999 & 1.000 & 1.000 \\
CMAPANK_L & 0.999 & 1.000 & 0.999 & 0.999 \\
CMAPKNE_R & 1.000 & 1.000 & 0.996 & 0.999 \\
CMAPANK_R & 0.997 & 0.999 & 0.998 & 0.998 \\
FWAVE_L & 0.996 & 0.996 & 0.993 & 0.995 \\
MCV_L & 0.994 & 0.991 & 0.996 & 0.994 \\
FWAVE_R & 0.996 & 0.993 & 0.992 & 0.994 \\
SSA_R & 0.983 & 0.998 & 0.996 & 0.993 \\
SPSA_L & 0.997 & 0.985 & 0.994 & 0.992 \\
MCV_R & 0.994 & 0.992 & 0.988 & 0.991 \\
NS & 0.988 & 0.988 & 0.979 & 0.985 \\
HBA1C & 0.986 & 0.968 & 0.992 & 0.982 \\
FEET_MEAN_ESC & 0.983 & 0.974 & 0.987 & 0.981 \\
SPSA_R & 0.983 & 0.973 & 0.985 & 0.980 \\
SSC_R & 0.978 & 0.979 & 0.982 & 0.980 \\
HAND_MEAN_ESC & 0.977 & 0.987 & 0.973 & 0.979 \\
SPSC_L & 0.975 & 0.977 & 0.980 & 0.977 \\
SSC_L & 0.963 & 0.973 & 0.989 & 0.975 \\
SPSC_R & 0.970 & 0.968 & 0.979 & 0.972 \\
DL_L & 0.967 & 0.966 & 0.977 & 0.970 \\
DL_R & 0.973 &

#### Counterfactual Instances Summary

In [98]:
print(ioi_summary_df.to_latex(index=False, float_format="{:.3f}".format))

\begin{tabular}{lllllll}
\toprule
Model & Instances & Misclassified & Instances with CF & Sparsity Mean & L1 Mean & L2 Mean \\
\midrule
Split 0 & 6 & 4 & 6 & 25.669 & 106.291 & 30.464 \\
Split 1 & 7 & 4 & 6 & 25.970 & 104.365 & 29.863 \\
Split 2 & 9 & 6 & 8 & 25.543 & 101.207 & 29.327 \\
\bottomrule
\end{tabular}



#### Models  Summary

In [109]:
print(cf_dfs_stacked.to_latex(index=False, float_format="{:.3f}".format))

\begin{tabular}{lrlllllrlll}
\toprule
model & ID & prediction & actual & misclassified & probability & margin & CF Count & Sparsity & L1 Mean & L2 Mean \\
\midrule
0 & 4 & Confirmed & Confirmed & No & 0.719 & 0.083 & 4 & 28.250 & 116.995 & 30.669 \\
0 & 55 & Confirmed & Confirmed & No & 0.767 & 0.131 & 22 & 27.545 & 107.345 & 29.713 \\
0 & 67 & Unconfirmed & Confirmed & Yes & 0.187 & 0.449 & 20 & 24.000 & 92.632 & 28.135 \\
0 & 106 & Unconfirmed & Confirmed & Yes & 0.524 & 0.111 & 4 & 25.250 & 114.363 & 34.028 \\
0 & 128 & Unconfirmed & Confirmed & Yes & 0.275 & 0.360 & 35 & 22.829 & 93.903 & 28.710 \\
0 & 158 & Unconfirmed & Confirmed & Yes & 0.037 & 0.598 & 35 & 26.143 & 112.511 & 31.527 \\
1 & 12 & Confirmed & Confirmed & No & 0.666 & 0.031 & 38 & 27.605 & 111.169 & 30.800 \\
1 & 54 & Confirmed & Confirmed & No & 0.751 & 0.116 & 45 & 28.267 & 121.284 & 32.891 \\
1 & 73 & Confirmed & Unconfirmed & Yes & 0.934 & 0.298 & 39 & 28.308 & 122.743 & 32.733 \\
1 & 94 & Unconfirmed & Unconfir